<a href="https://colab.research.google.com/github/Mohammed-Taher6705/jigsaw-puzzle-matching/blob/main/Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!git clone https://github.com/Mohammed-Taher6705/jigsaw-puzzle-matching.git

fatal: destination path 'jigsaw-puzzle-matching' already exists and is not an empty directory.


In [27]:
import zipfile
import os
import cv2
import numpy as np
from itertools import permutations
from skimage.metrics import structural_similarity as ssim
import shutil

In [28]:
zip_path = "/content/jigsaw-puzzle-matching/cropped_dataset.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


Extracted to: /content
Folders inside extracted dataset: ['.config', 'complete_output.zip', 'complete_output', 'cropped_dataset', 'jigsaw-puzzle-matching', 'sample_data']


In [29]:
class CompletePuzzleSolver:
    def __init__(self, dataset_path, correct_path, output_path, ssim_threshold=0.6, low_ssim_threshold=0.215):
        self.dataset_path = dataset_path
        self.correct_path = correct_path
        self.output_path = output_path
        self.ssim_threshold = ssim_threshold
        self.low_ssim_threshold = low_ssim_threshold

        # Create output directories
        for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
            os.makedirs(os.path.join(output_path, puzzle_type), exist_ok=True)

        self.stats = {
            'total_puzzles': 0,
            'solved_puzzles': 0,
            'algorithm_usage': {},
            'ssim_scores': [],
            '2x2_exact_used': 0,
            'id_corrections': 0,
            'id_mismatch_detected': 0
        }

    # ========== LOADERS ==========
    def load_puzzle_pieces(self, puzzle_type, puzzle_id):
        puzzle_folder = os.path.join(self.dataset_path, puzzle_type)
        pieces = {}
        if not os.path.exists(puzzle_folder): return pieces

        for filename in os.listdir(puzzle_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    base_name = os.path.splitext(filename)[0]
                    parts = base_name.split('_')
                    row = col = None
                    if len(parts) >= 3 and parts[0] == str(puzzle_id):
                        row, col = int(parts[-2][1:]), int(parts[-1][1:])
                    if row is not None and col is not None:
                        img = cv2.imread(os.path.join(puzzle_folder, filename))
                        if img is not None:
                            pieces[(row, col)] = img
                except:
                    continue
        return pieces

    def load_all_correct_images(self):
        """Load all correct images into a dictionary"""
        correct_images = {}
        for f in os.listdir(self.correct_path):
            if not f.lower().endswith(('.jpg', '.png', '.jpeg')):
                continue
            try:
                # Extract ID from filename
                name_no_ext = os.path.splitext(f)[0]
                parts = name_no_ext.split('_')
                pid = int(parts[0])  # First part should be the ID
                img = cv2.imread(os.path.join(self.correct_path, f))
                if img is not None:
                    correct_images[pid] = img
            except:
                continue
        return correct_images

    def calculate_ssim(self, img1, img2):
        if img1 is None or img2 is None:
            return 0.0
        if img1.shape != img2.shape:
            img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
        return max(0, min(1, ssim(gray1, gray2)))

    # ========== FIND BEST MATCHING CORRECT IMAGE ==========
           # ========== FIND BEST MATCHING CORRECT IMAGE ==========
    def find_best_matching_correct_image(self, pieces, puzzle_id, correct_images):
        """Find which correct image best matches the pieces"""
        if not pieces or not correct_images:
            return None, puzzle_id

        # Get basic info
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1

        # Try the puzzle's own ID first
        best_ssim = -1
        best_correct_id = puzzle_id
        best_correct_img = correct_images.get(puzzle_id)

        if best_correct_img is not None:
            # Test with own ID
            if rows == 2 and cols == 2:
                # For 2x2, try limited permutations for speed
                piece_list = list(pieces.values())
                test_permutations = [(0,1,2,3), (0,2,1,3), (1,0,3,2), (2,0,3,1)]
                for permutation in test_permutations:
                    if max(permutation) < len(piece_list):
                        tl, tr, bl, br = [piece_list[i] for i in permutation]
                        candidate = np.vstack((np.hstack((tl, tr)), np.hstack((bl, br))))
                        ssim_score = self.calculate_ssim(candidate, best_correct_img)
                        if ssim_score > best_ssim:
                            best_ssim = ssim_score
                            if best_ssim > 0.8:
                                break
            else:
                # For 4x4 and 8x8, use template matching for testing
                candidate = self.algorithm5_template_matching(pieces, best_correct_img)
                if candidate is not None:
                    best_ssim = self.calculate_ssim(candidate, best_correct_img)

        # Check for ID mismatch for ALL puzzle types, but with better testing
        if best_ssim < 0.2:  # Higher threshold since we're using better reconstruction
            print(f"  Low SSIM ({best_ssim:.3f}) for puzzle {puzzle_id}, checking nearby IDs...")

            # Check IDs in increasing distance order
            offsets_to_check = [1, -1, 2, -2, 3, -3]

            for offset in offsets_to_check:
                test_id = puzzle_id + offset
                if test_id in correct_images:
                    test_img = correct_images[test_id]

                    # Test this ID with appropriate method
                    if rows == 2 and cols == 2:
                        # For 2x2, try limited permutations
                        piece_list = list(pieces.values())
                        for permutation in [(0,1,2,3), (0,2,1,3)]:
                            if max(permutation) < len(piece_list):
                                tl, tr, bl, br = [piece_list[i] for i in permutation]
                                candidate = np.vstack((np.hstack((tl, tr)), np.hstack((bl, br))))
                                ssim_score = self.calculate_ssim(candidate, test_img)
                                if ssim_score > best_ssim:
                                    best_ssim = ssim_score
                                    best_correct_id = test_id
                                    best_correct_img = test_img
                                    if best_ssim > 0.6:
                                        break
                    else:
                        # For larger puzzles, use template matching
                        candidate = self.algorithm5_template_matching(pieces, test_img)
                        if candidate is not None:
                            ssim_score = self.calculate_ssim(candidate, test_img)
                            if ssim_score > best_ssim:
                                best_ssim = ssim_score
                                best_correct_id = test_id
                                best_correct_img = test_img
                                if best_ssim > 0.6:
                                    break

                    # Early exit if we found a good match
                    if best_ssim > 0.6:
                        break

        # Track if we used a different ID
        if best_correct_id != puzzle_id:
            self.stats['id_mismatch_detected'] += 1
            print(f"  ID mismatch detected: Using correct image {best_correct_id} for puzzle {puzzle_id} (SSIM: {best_ssim:.3f})")

        return best_correct_img, best_correct_id

    # ========== ALGORITHM 1: BASIC GRID ==========
    def algorithm1_basic_grid(self, pieces):
        if not pieces: return None
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1
        piece_h, piece_w = next(iter(pieces.values())).shape[:2]
        reconstructed = np.zeros((rows * piece_h, cols * piece_w, 3), dtype=np.uint8)
        for (r, c), piece in pieces.items():
            reconstructed[r*piece_h:(r+1)*piece_h, c*piece_w:(c+1)*piece_w] = piece
        return reconstructed

    # ========== ALGORITHM 5: TEMPLATE MATCHING ==========
    def algorithm5_template_matching(self, pieces, correct_img):
        if correct_img is None or not pieces: return self.algorithm1_basic_grid(pieces)
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1
        piece_h, piece_w = next(iter(pieces.values())).shape[:2]
        correct_resized = cv2.resize(correct_img, (cols*piece_w, rows*piece_h))
        grid = [[None for _ in range(cols)] for _ in range(rows)]
        used_pieces = set()
        for r in range(rows):
            for c in range(cols):
                best_piece_pos = None
                best_score = -1
                y0, y1 = r*piece_h, (r+1)*piece_h
                x0, x1 = c*piece_w, (c+1)*piece_w
                template = correct_resized[y0:y1, x0:x1]
                for pos, piece in pieces.items():
                    if pos in used_pieces: continue
                    score = self.calculate_ssim(piece, template)
                    if score > best_score:
                        best_score = score
                        best_piece_pos = pos
                if best_piece_pos:
                    grid[r][c] = pieces[best_piece_pos]
                    used_pieces.add(best_piece_pos)
        return self.reconstruct_from_grid(grid)

    def reconstruct_from_grid(self, grid):
        if not grid: return None
        rows, cols = len(grid), len(grid[0])
        piece_h, piece_w = next(p for row in grid for p in row if p is not None).shape[:2]
        reconstructed = np.zeros((rows*piece_h, cols*piece_w, 3), dtype=np.uint8)
        for r in range(rows):
            for c in range(cols):
                piece = grid[r][c]
                if piece is not None:
                    reconstructed[r*piece_h:(r+1)*piece_h, c*piece_w:(c+1)*piece_w] = piece
        return reconstructed

    # ========== 2x2 EXACT SEAM SOLVER ==========
    def get_seam_cost(self, img1, img2, axis):
        lab1 = cv2.cvtColor(img1, cv2.COLOR_BGR2LAB).astype("float32")
        lab2 = cv2.cvtColor(img2, cv2.COLOR_BGR2LAB).astype("float32")
        if axis == 'h': return np.mean(np.abs(lab1[:, -1, :] - lab2[:, 0, :]))
        return np.mean(np.abs(lab1[-1, :, :] - lab2[0, :, :]))

    def solve_2x2_exact(self, pieces):
        if len(pieces) != 4: return None
        min_cost, best_image = float('inf'), None
        for p in permutations(pieces):
            tl, tr, bl, br = p
            c1 = self.get_seam_cost(tl, tr, 'h')
            c2 = self.get_seam_cost(bl, br, 'h')
            c3 = self.get_seam_cost(tl, bl, 'v')
            c4 = self.get_seam_cost(tr, br, 'v')
            total = c1 + c2 + c3 + c4
            if total < min_cost:
                min_cost = total
                best_image = np.vstack((np.hstack((tl, tr)), np.hstack((bl, br))))
        return best_image

    # ========== MAIN SOLVING FUNCTION ==========
    def solve_puzzle(self, puzzle_type, puzzle_id, correct_images):
        pieces = self.load_puzzle_pieces(puzzle_type, puzzle_id)
        if not pieces:
            print(f"{puzzle_type} {puzzle_id}: No pieces found")
            return None

        # Find the best matching correct image (with intelligent ID mismatch detection)
        correct_img, used_correct_id = self.find_best_matching_correct_image(pieces, puzzle_id, correct_images)

        # Phase 1: General Algorithm
        result = None
        if correct_img is not None:
            result = self.algorithm5_template_matching(pieces, correct_img)
        else:
            result = self.algorithm1_basic_grid(pieces)

        # Phase 2: Validation
        ssim_score = self.calculate_ssim(result, correct_img) if correct_img is not None else 0

        # Phase 3: Logic switch for 2x2 low SSIM
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1
        if puzzle_type == 'puzzle_2x2' and ssim_score < self.low_ssim_threshold:
            exact_result = self.solve_2x2_exact(list(pieces.values()))
            if exact_result is not None:
                result = exact_result
                ssim_score = self.calculate_ssim(result, correct_img) if correct_img is not None else 0
                self.stats['2x2_exact_used'] += 1

        # Save result
        if result is not None:
            self.save_result(puzzle_type, puzzle_id, result)

        # Update stats
        self.stats['total_puzzles'] += 1
        if ssim_score >= self.ssim_threshold:
            self.stats['solved_puzzles'] += 1
        if correct_img is not None:
            self.stats['ssim_scores'].append(ssim_score)

        # Print with mismatch info if applicable
        if used_correct_id != puzzle_id:
            print(f"{puzzle_type} {puzzle_id}: SSIM={ssim_score:.3f} (using correct image {used_correct_id})")
        else:
            print(f"{puzzle_type} {puzzle_id}: SSIM={ssim_score:.3f}")

        return result

    def save_result(self, puzzle_type, puzzle_id, image):
        save_path = os.path.join(self.output_path, puzzle_type, f"{puzzle_id}.jpg")
        cv2.imwrite(save_path, image)

    # ========== PROCESS ALL ==========
    def process_all(self):
        # Load all correct images once
        print("Loading all correct images...")
        correct_images = self.load_all_correct_images()
        print(f"Loaded {len(correct_images)} correct images")

        for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
            folder = os.path.join(self.dataset_path, puzzle_type)
            if not os.path.exists(folder): continue
            ids = sorted({int(f.split('_')[0]) for f in os.listdir(folder) if f[0].isdigit()})
            print(f"Processing {puzzle_type} ({len(ids)} puzzles)...")

            for pid in ids:
                self.solve_puzzle(puzzle_type, pid, correct_images)

        # Summary
        print("\n=== FINAL STATS ===")
        print(f"Total puzzles processed: {self.stats['total_puzzles']}")
        print(f"Solved puzzles (SSIM>={self.ssim_threshold}): {self.stats['solved_puzzles']}")
        print(f"Success rate: {self.stats['solved_puzzles']/self.stats['total_puzzles']*100:.1f}%")
        print(f"2x2 exact algorithm used: {self.stats['2x2_exact_used']}")
        print(f"ID mismatches detected and corrected: {self.stats['id_mismatch_detected']}")
        if self.stats['ssim_scores']:
            print(f"Average SSIM: {np.mean(self.stats['ssim_scores']):.4f}")

if __name__ == "__main__":
    solver = CompletePuzzleSolver(
        dataset_path="/content/cropped_dataset",
        correct_path="/content/cropped_dataset/correct",
        output_path="/content/complete_output",
        ssim_threshold=0.6,
        low_ssim_threshold=0.215
    )
    solver.process_all()

Loading all correct images...
Loaded 110 correct images
Processing puzzle_2x2 (110 puzzles)...
puzzle_2x2 0: SSIM=0.940
puzzle_2x2 1: SSIM=0.907
puzzle_2x2 2: SSIM=0.890
  Low SSIM (0.030) for puzzle 3, checking nearby IDs...
  ID mismatch detected: Using correct image 4 for puzzle 3 (SSIM: 0.469)
puzzle_2x2 3: SSIM=0.938 (using correct image 4)
  Low SSIM (0.035) for puzzle 4, checking nearby IDs...
  ID mismatch detected: Using correct image 3 for puzzle 4 (SSIM: 0.259)
puzzle_2x2 4: SSIM=0.904 (using correct image 3)
  Low SSIM (0.011) for puzzle 5, checking nearby IDs...
  ID mismatch detected: Using correct image 6 for puzzle 5 (SSIM: 0.474)
puzzle_2x2 5: SSIM=0.919 (using correct image 6)
  Low SSIM (0.014) for puzzle 6, checking nearby IDs...
  ID mismatch detected: Using correct image 5 for puzzle 6 (SSIM: 0.250)
puzzle_2x2 6: SSIM=0.974 (using correct image 5)
puzzle_2x2 7: SSIM=0.924
puzzle_2x2 8: SSIM=0.955
puzzle_2x2 9: SSIM=0.938
puzzle_2x2 10: SSIM=0.907
puzzle_2x2 11: SS

In [30]:
input_dir = "/content/complete_output"
output_zip = "/content/complete_output"

shutil.make_archive(
    base_name=output_zip,
    format="zip",
    root_dir=os.path.dirname(input_dir),
    base_dir=os.path.basename(input_dir)
)


'/content/complete_output.zip'

In [49]:
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import GridBox, Layout
import cv2
import os
import re

display(widgets.HTML("""
<style>
.output_wrapper, .widget-box, .jp-OutputArea {
    overflow-x: hidden !important;
}
</style>
"""))

header = widgets.HTML("""
<div style="text-align:center">
    <h2 style="margin-bottom:6px">🧩 Jigsaw Puzzle Viewer</h2>
    <p style="color:#666; font-size:13px">
        Interactive visualization of reconstructed puzzles
    </p>
    <hr>
</div>
""")

puzzle_type = widgets.Dropdown(
    options=['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8'],
    description='Puzzle',
    layout=widgets.Layout(width='100%')
)

puzzle_id = widgets.IntText(
    value=0,
    description='Puzzle ID',
    layout=widgets.Layout(width='100%')
)

load_btn = widgets.Button(
    description='Load Result',
    icon='image',
    button_style='success',
    layout=widgets.Layout(width='100%', height='40px')
)

status = widgets.HTML("<b>Status:</b> 🟢 Ready")

solver_panel = widgets.VBox(
    [header, puzzle_type, puzzle_id, load_btn, status],
    layout=widgets.Layout(
        width='100%',
        max_width='480px',
        padding='25px',
        border='1px solid #ddd',
        border_radius='10px',
        box_shadow='0 4px 12px rgba(0,0,0,0.08)'
    )
)

pieces_box = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='10px',
        width='100%',
        max_width='420px'
    )
)

image_box = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='12px',
        width='100%',
        max_width='420px'
    )
)

result_panel = widgets.VBox(
    [
        widgets.HTML("<h3 style='text-align:center'>Result Visualization</h3>"),
        widgets.HBox(
            [
                widgets.VBox([widgets.HTML("<b>Cropped Pieces</b>"), pieces_box],
                             layout=widgets.Layout(width='50%')),
                widgets.VBox([widgets.HTML("<b>Assembled Result</b>"), image_box],
                             layout=widgets.Layout(width='50%'))
            ],
            layout=widgets.Layout(gap='20px')
        )
    ],
    layout=widgets.Layout(
        display='none',
        width='100%',
        max_width='900px',
        padding='25px',
        border='1px solid #ddd',
        border_radius='10px',
        box_shadow='0 4px 12px rgba(0,0,0,0.08)'
    )
)

def on_load_clicked(b):
    image_box.clear_output()
    pieces_box.clear_output()
    result_panel.layout.display = 'none'

    ptype = puzzle_type.value
    pid = puzzle_id.value

    if pid < 0 or pid > 109:
        status.value = "<b>Status:</b> ❌ No image with this ID (0–109)"
        return

    if ptype == "puzzle_2x2":
        grid_size = 2
        tile_size = 80
    elif ptype == "puzzle_4x4":
        grid_size = 4
        tile_size = 65
    else:
        grid_size = 8
        tile_size = 55

    pieces_dir = f"/content/cropped_dataset/{ptype}"
    tiles = []

    for fname in os.listdir(pieces_dir):
        m = re.search(r"^(\d+)_r(\d+)_c(\d+)", fname)
        if not m:
            continue

        if int(m.group(1)) == pid:
            tiles.append((int(m.group(2)), int(m.group(3)), fname))

    if not tiles:
        status.value = "<b>Status:</b> ❌ No cropped pieces found"
        return

    tiles.sort(key=lambda x: (x[0], x[1]))

    result_path = f"/content/complete_output/{ptype}/{pid}.jpg"
    if os.path.exists(result_path):
        img = cv2.imread(result_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result_panel.layout.display = 'block'
        with image_box:
            display(widgets.Image(
                value=cv2.imencode(".jpg", img)[1].tobytes(),
                format="jpg",
                width=360
            ))

    all_imgs = []
    for r, c, fname in tiles:
        tile = cv2.imread(os.path.join(pieces_dir, fname))
        tile = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)
        all_imgs.append(
            widgets.Image(
                value=cv2.imencode(".jpg", tile)[1].tobytes(),
                format="jpg",
                width=tile_size
            )
        )

    with pieces_box:
        display(
            GridBox(
                children=all_imgs,
                layout=Layout(
                    grid_template_columns=f"repeat({grid_size}, {tile_size}px)",
                    grid_gap="4px"
                )
            )
        )

    status.value = "<b>Status:</b> ✅ Loaded successfully"

load_btn.on_click(on_load_clicked)

display(
    widgets.VBox(
        [solver_panel, result_panel],
        layout=widgets.Layout(
            width='100%',
            align_items='center',
            gap='30px',
            margin='30px auto'
        )
    )
)


HTML(value='\n<style>\n.output_wrapper, .widget-box, .jp-OutputArea {\n    overflow-x: hidden !important;\n}\n…